# Resume d? ?n tr?n Google Drive m?i

Notebook n?y d?ng ?? **ki?m tra v? ti?p t?c**, kh?ng ch?y l?i 75 cell c?. N? kh?ng ch?a token v? kh?ng t? x?a d? li?u.

Tr??c khi ch?y:
1. M? notebook b?ng t?i kho?n Google m?i.
2. T? Drive c?, share th? m?c `AI_Project_Insurance` v? th?m shortcut v?o My Drive m?i v?i t?n `AI_Project_Insurance_OLD`.
3. Chu?n b? file ZIP code/report do Codex t?o.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path

OLD_PROJECT = Path("/content/drive/MyDrive/AI_Project_Insurance_OLD")
NEW_PROJECT = Path("/content/drive/MyDrive/AI_Project_Insurance")

print("Old project exists:", OLD_PROJECT.exists(), OLD_PROJECT)
print("New project:", NEW_PROJECT)

if not OLD_PROJECT.exists():
    raise FileNotFoundError(
        "Kh?ng th?y shortcut Drive c?. H?y s?a OLD_PROJECT theo t?n shortcut trong MyDrive."
    )


## Upload g?i code m?i

Ch?n file `AI_Project_Insurance_code_reports_2026-10-07.zip`. File n?y kh?ng ch?a raw data ho?c model artifact.


In [ ]:
from google.colab import files
import zipfile

uploaded = files.upload()
zip_files = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_files) != 1:
    raise RuntimeError("H?y upload ??ng m?t file ZIP code/report.")

bundle = Path("/content") / zip_files[0]
bundle.write_bytes(uploaded[zip_files[0]])
NEW_PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(bundle) as archive:
    archive.extractall(NEW_PROJECT)

print("?? gi?i n?n code v?o:", NEW_PROJECT)


## Ki?m tra tr?ng th?i th?t tr?n Drive c?

Cell n?y ch? ??c. K?t qu? gi?p ph?n bi?t conversion v1 v? v2.


In [ ]:
import json
import subprocess

checker = NEW_PROJECT / "scripts/check_cardd_resume_state.py"
completed = subprocess.run(
    ["python", str(checker), str(OLD_PROJECT)],
    check=True,
    capture_output=True,
    text=True,
)
state = json.loads(completed.stdout)
print(json.dumps(state, ensure_ascii=False, indent=2))


In [ ]:
expected_v2 = {"train": 1966, "val": 431, "test": 419}
actual_splits = state["conversion"]["reported_split_images"]

print("Bronze ready:", state["bronze"]["ready"])
print("Conversion metadata ready:", state["conversion"]["metadata_ready"])
print("Actual splits:", actual_splits)
print("Expected v2:", expected_v2)

if not state["bronze"]["ready"]:
    raise RuntimeError("Bronze ch?a ho?n ch?nh; d?ng migration.")
if not state["conversion"]["metadata_ready"]:
    raise RuntimeError("Labels/manifest/report ch?a ??ng b?; d?ng migration.")
if actual_splits != expected_v2:
    raise RuntimeError(
        "Drive c? kh?ng ch?a conversion v2 mong ??i. G?i output checker cho Codex tr??c khi ti?p t?c."
    )

print("Resume check: PASS")


## Copy ph?n c?n thi?t sang Drive m?i

Cell n?y copy labels v? metadata nh?. N? **kh?ng copy** th? m?c ?nh Processed. Bronze v?n ???c ??c qua shortcut c? trong b??c ??u.


In [ ]:
import shutil

old_processed = OLD_PROJECT / "data/processed/cardd_yolo_v1"
new_processed = NEW_PROJECT / "data/processed/cardd_yolo_v1"
new_processed.mkdir(parents=True, exist_ok=True)

shutil.copytree(
    old_processed / "labels",
    new_processed / "labels",
    dirs_exist_ok=True,
)
for filename in ["conversion_report.json", "split_manifest.jsonl"]:
    shutil.copy2(old_processed / filename, new_processed / filename)

old_reports = OLD_PROJECT / "reports"
if old_reports.is_dir():
    shutil.copytree(old_reports, NEW_PROJECT / "reports", dirs_exist_ok=True)

print("?? copy labels, manifest v? reports. Kh?ng copy ?nh Processed.")


## D?ng dataset t?m v? validate

C?c symlink n?m trong `/content`, n?n kh?ng l?m t?ng dung l??ng Drive m?i.


In [ ]:
import shutil

SOURCE_CARDD = OLD_PROJECT / "data/bronze/layerA_cardd"
TEMP_DATASET = Path("/content/cardd_yolo_v1")
if TEMP_DATASET.exists():
    shutil.rmtree(TEMP_DATASET)

subprocess.run(
    [
        "python",
        str(NEW_PROJECT / "scripts/materialize_cardd_training_view.py"),
        str(SOURCE_CARDD),
        str(new_processed),
        str(TEMP_DATASET),
    ],
    check=True,
)

validation_path = Path("/content/cardd_yolo_validation_v2.json")
subprocess.run(
    [
        "python",
        str(NEW_PROJECT / "scripts/validate_yolo_seg_dataset.py"),
        str(TEMP_DATASET),
        "--output",
        str(validation_path),
    ],
    check=True,
)

validation = json.loads(validation_path.read_text(encoding="utf-8"))
print("Validation valid:", validation["valid"])
print("Training YAML:", TEMP_DATASET / "data.yaml")


## ?i?m d?ng

N?u `Validation valid: True`, d? li?u k? thu?t ?? s?n s?ng ?? ti?p t?c. Ch?a ch?y training trong notebook n?y. Tr??c khi train CarDD, l?u b?ng ch?ng prior consent c?a PIC Lab v? d?ng notebook training ri?ng.
